In [ ]:
!pip install /home/satiy/vaderSentiment-3.3.2-py2.py3-none-any.whl

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import lit, udf, col
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from pyspark.sql.types import StringType, FloatType, StructType, StructField


In [2]:
spark = SparkSession.builder.appName("ReadFromGCS").getOrCreate()

25/04/15 12:08:59 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [ ]:
#read the parquet files
crypto_prices = spark.read.parquet('gs://zoomcamp_project_v1/crypto_data/crypto_prices/*')

#register as temp table
crypto_prices.registerTempTable('crypto_prices')

#extract the result
merged_df = spark.sql("""
WITH data AS 
(
    SELECT
        *,
        row_number() over(partition by timestamp order by ingested_at desc) AS row_num
    FROM
        crypto_prices
)
SELECT * FROM data WHERE row_num=1;
""")

In [ ]:
merged_df.count()

In [ ]:
merged_df.head(5)

In [ ]:
merged_df.write.parquet('gs://zoomcamp_project_v1/crypto_data/crypto_prices_merged', mode='overwrite')

# Read social media comments

In [ ]:
columns_selected = ["post_id", "post_title", "post_timestamp", "comment_id", "comment_text", "comment_date", "social_media"]

In [ ]:
reddit_df = spark.read.parquet("gs://zoomcamp_project_v1/crypto_sentiments/crypto_sentiments/")\
                      .withColumn('social_media', lit('reddit')) \
                      .dropDuplicates(['post_timestamp', 'post_title', 'comment_text', 'comment_date']) \
                      .filter("comment_text != '' ").filter("post_title != '' ")\
                      .select(columns_selected)

In [ ]:
reddit_df.count()

In [ ]:
cryptopanic_df = spark.read.parquet("gs://zoomcamp_project_v1/crypto_sentiments/cryptopanic_sentiments/")\
                           .withColumn('social_media', lit('cryptopanic')) \
                           .dropDuplicates(['post_timestamp', 'post_title', 'comment_text', 'comment_date']) \
                           .filter("comment_text != '' ").filter("post_title != '' ")\
                           .select(columns_selected)

In [ ]:
cryptopanic_df.count()

In [ ]:
reddit_df.printSchema()

In [ ]:
cryptopanic_df.printSchema()

In [ ]:
cryptosentiments_df = reddit_df.unionAll(cryptopanic_df)

In [ ]:
cryptosentiments_df.write.parquet('gs://zoomcamp_project_v1/crypto_sentiments/crypto_sentiments_merged', mode='overwrite')

In [3]:
cryptosentiments_df = spark.read.parquet('gs://zoomcamp_project_v1/crypto_sentiments/crypto_sentiments_merged')

In [4]:
prices_df = spark.read.parquet('gs://zoomcamp_project_v1/crypto_data/crypto_prices_merged')

In [5]:
cryptosentiments_df.printSchema()

root
 |-- post_id: long (nullable = true)
 |-- post_title: string (nullable = true)
 |-- post_timestamp: timestamp (nullable = true)
 |-- comment_id: long (nullable = true)
 |-- comment_text: string (nullable = true)
 |-- comment_date: timestamp (nullable = true)
 |-- social_media: string (nullable = true)



In [6]:
cryptosentiments_df.count()

33340

In [7]:
cryptosentiments_df.show(5)

+-------+--------------------+--------------------+----------+--------------------+--------------------+------------+
|post_id|          post_title|      post_timestamp|comment_id|        comment_text|        comment_date|social_media|
+-------+--------------------+--------------------+----------+--------------------+--------------------+------------+
|    104|Just became owner...|2024-12-30 16:41:...|        21|0.25 Bitcoin is 2...|2024-12-30 17:08:...|      reddit|
|    104|Just became owner...|2024-12-30 16:41:...|        72|0.25 is actually ...|2024-12-30 20:56:...|      reddit|
|    104|Just became owner...|2024-12-30 16:41:...|       119|A quarter BTC is ...|2024-12-31 09:54:...|      reddit|
|    104|Just became owner...|2024-12-30 16:41:...|        39|At least someone ...|2024-12-30 17:55:...|      reddit|
|    104|Just became owner...|2024-12-30 16:41:...|       111|Being proud when ...|2024-12-31 05:45:...|      reddit|
+-------+--------------------+--------------------+-----

In [10]:

# Initialize VADER Sentiment Analyzer
analyzer = SentimentIntensityAnalyzer()

# Define a UDF to apply VADER sentiment analysis
def get_sentiment(text):
    if not text or len(text.strip()) == 0:
        return ("NEUTRAL", 0.0)
    score = analyzer.polarity_scores(text)['compound']
    if score > 0.05:
        return ("POSITIVE", score)
    elif score < -0.05:
        return ("NEGATIVE", score)
    else:
        return ("NEUTRAL", score)

# Register UDF
sentiment_udf = udf(get_sentiment, StructType([StructField("label", StringType(), True), StructField("score", FloatType(), True)]))


# Apply UDF to get sentiment
df_with_sentiment = cryptosentiments_df.withColumn("post_sentiment", sentiment_udf(cryptosentiments_df["post_title"])) \
                                       .withColumn("comment_sentiment", sentiment_udf(cryptosentiments_df["comment_text"]))

# Extract sentiment labels and scores
cols = ['post_id','post_title','post_timestamp','comment_id','comment_text','comment_date',
        'social_media','post_label','post_score','comment_label','comment_score']

df_sentiments = df_with_sentiment.withColumn("post_label", col("post_sentiment.label"))\
                                 .withColumn("post_score", col("post_sentiment.score"))\
                                 .withColumn("comment_label", col("comment_sentiment.label"))\
                                 .withColumn("comment_score", col("comment_sentiment.score"))\
                                 .select(cols)


# Sentiment metric derivation


In [11]:
df_sentiments.registerTempTable('sentiments')
prices_df.registerTempTable('prices')

/usr/lib/spark/python/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [12]:
spark.sql('SELECT * FROM sentiments').count()

33340

In [13]:
#extract the result
df_sentiments_processed = spark.sql("""

SELECT
    hour_timestamp,
    post_title,
    post_timestamp,
    comment_text,
    comment_date As comment_timestamp,
    weighted_post_sentiment,
    weighted_comment_sentiment,
    COUNT(comment_text) OVER (PARTITION BY post_title, post_timestamp, hour_timestamp) AS velocity
FROM
    (
        SELECT
            *,
            post_score*(1-((post_score-comment_score)/(1 + (post_score-comment_score)))) AS weighted_post_sentiment,
            comment_score*((post_score-comment_score)/(1 + (post_score-comment_score))) AS weighted_comment_sentiment,
            date_trunc("Hour",post_timestamp) AS hour_timestamp
        FROM sentiments
    )

""")

# Price Volatility 

In [14]:
df_prices_processsed = spark.sql("""

SELECT 
    hour_timestamp,
    measured_timestamp,
    price,
    market_cap,
    volume,
    stddev(price) OVER (PARTITION BY daily_timestamp) AS daily_volatility
FROM
(
    SELECT 
        date_trunc("Hour",timestamp) AS hour_timestamp,
        date_trunc("Day",timestamp) AS daily_timestamp,
        timestamp as measured_timestamp,
        price,
        market_cap,
        volume
    FROM prices
);

""")

In [17]:
df_sentiments_processed.registerTempTable('sentiments_processed')
df_prices_processsed.registerTempTable('prices_processed')

In [18]:
combined_results = spark.sql("""

WITH aggregated_sentiments AS
(
    SELECT 
        hour_timestamp,
        AVG(weighted_post_sentiment) AS avg_weighted_post_sentiment,
        AVG(weighted_comment_sentiment) AS avg_weighted_comment_sentiment,
        AVG(velocity) AS avg_comment_velocity
    FROM sentiments_processed
    GROUP BY hour_timestamp
),
aggregated_prices AS
(
    SELECT 
        hour_timestamp,
        AVG(price) as avg_price,
        AVG(market_cap) AS avg_market_cap,
        AVG(volume) AS avg_volume,
        AVG(daily_volatility) AS daily_volatility
    FROM prices_processed
    GROUP BY hour_timestamp
)
SELECT
    A.hour_timestamp AS hour_timestamp,
    A.avg_price AS avg_price,
    A.avg_market_cap AS avg_market_cap,
    A.avg_volume AS avg_volume,   
    A.daily_volatility AS daily_volatility,
    B.avg_weighted_post_sentiment AS avg_weighted_post_sentiment,
    B.avg_weighted_comment_sentiment AS avg_weighted_comment_sentiment,
    B.avg_comment_velocity AS avg_comment_velocity
FROM
    aggregated_prices A
    JOIN aggregated_sentiments B
    ON A.hour_timestamp=B.hour_timestamp
ORDER BY A.hour_timestamp
""")

In [22]:
# Use the Cloud Storage bucket for temporary BigQuery export data used
# by the connector.
bucket = "zoomcamp_project_v1"
spark.conf.set('temporaryGcsBucket', bucket)

# Save the data to BigQuery
df_sentiments_processed.write.format('bigquery') \
  .option('table', 'crypto_project.sentiments') \
  .save()

# Save the data to BigQuery
df_prices_processsed.write.format('bigquery') \
  .option('table', 'crypto_project.prices') \
  .save()

# Save the data to BigQuery
combined_results.write.format('bigquery') \
  .option('table', 'crypto_project.hourly_aggregated_data') \
  .save()

25/04/15 18:18:27 WARN DAGScheduler: Broadcasting large task binary with size 1103.5 KiB
25/04/15 18:18:47 WARN DAGScheduler: Broadcasting large task binary with size 1155.4 KiB
